# Benchmarking ECDLP functions

In [14]:
import qrisp
import numpy as np
import src.classical.ec_arithmetic as clECarithm
import src.quantum.ec_arithmetic as qECarithm

In [15]:
def t_depth_indicator(op):
    if op.name in ["t", "t_dg"]:
        return 1
    else:
        return 0

def benchmark_circuit(res):
    
    compiled_circuit = res.qs.compile(compile_mcm = True, gate_speed = t_depth_indicator, workspace = res.size)
    qubit_count = compiled_circuit.num_qubits()
    gate_counts = compiled_circuit.transpile().count_ops()
    #print(gate_counts)
    cnot_count = gate_counts.get('cx', 0) + 0.5*gate_counts.get('c_if_cz', 0)
    t_count = gate_counts.get('t', 0) + gate_counts.get('t_dg', 0)
    t_depth = compiled_circuit.depth(depth_indicator = t_depth_indicator)  # Assumes depth of T gates aligns with circuit depth
    return {
        "qubit_count": qubit_count,
        "t_count": t_count,
        "cnot_count": cnot_count,
        "t_depth": t_depth
    }

In [16]:
p=7
a=5
b=4
curve = clECarithm.EllCurve(a, b, p)

In [17]:
benchmark_results_kaliski = {}
for val in range(1,p):
    v = qrisp.QuantumModulus(p, inpl_adder=qrisp.gidney_adder)
    v[:] = val
    m = qrisp.QuantumArray(qtype=qrisp.QuantumBool(), shape=(2 * p.bit_length(),))

    res_kaliski = qECarithm.kaliski_quantum(v, p, m)
    for a in m:
        a.delete()
    benchmark_results_kaliski[val] = benchmark_circuit(res_kaliski)


{'x': 521, 'cx': 5440, 'h': 1638, 't': 1378, 't_dg': 1540, 's': 422, 'measure': 422, 'c_if_cz': 422, 'reset': 422}
{'x': 521, 'cx': 5440, 'h': 1638, 't': 1378, 't_dg': 1540, 's': 422, 'measure': 422, 'c_if_cz': 422, 'reset': 422}
{'x': 522, 'cx': 5440, 'h': 1638, 't': 1378, 't_dg': 1540, 's': 422, 'measure': 422, 'c_if_cz': 422, 'reset': 422}
{'x': 521, 'cx': 5440, 'h': 1638, 't': 1378, 't_dg': 1540, 's': 422, 'measure': 422, 'c_if_cz': 422, 'reset': 422}
{'x': 522, 'cx': 5440, 'h': 1638, 't': 1378, 't_dg': 1540, 's': 422, 'measure': 422, 'c_if_cz': 422, 'reset': 422}
{'x': 522, 'cx': 5440, 'h': 1638, 't': 1378, 't_dg': 1540, 's': 422, 'measure': 422, 'c_if_cz': 422, 'reset': 422}


In [19]:
# Display results in a readable format
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_kaliski, orient="index")
results_df.index.name = "Value of v"
print(results_df)

            qubit_count  t_count  cnot_count  t_depth
Value of v                                           
1                    30     2918      5651.0      855
2                    30     2918      5651.0      855
3                    30     2918      5651.0      855
4                    30     2918      5651.0      855
5                    30     2918      5651.0      855
6                    30     2918      5651.0      855


In [ ]:
anc_values = [
    (2, 6),
    (4, 2),
    (0, 5),
    (5, 0),
]

x1 = qrisp.QuantumModulus(2**(p.bit_length()))
x1[:] = 1
mod_p = qrisp.QuantumModulus(p, inpl_adder=qrisp.gidney_adder)
G = [3, 2]

benchmark_results_ec_add = {}

for anc_pair in anc_values:
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    anc[:] = anc_pair
    res_ec_add = qECarithm.qrisp_ell_add_inpl(anc, G, p)
    bm = benchmark_circuit(res_ec_add[0])
    benchmark_results_ec_add[tuple(anc_pair)] = bm

In [ ]:
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_ec_add, orient="index")
results_df.index.name = "Ancilla Values (anc)"
print(results_df)

In [ ]:
x1 = qrisp.QuantumModulus(2**(p.bit_length()))
x1[:] = 1
mod_p = qrisp.QuantumModulus(p, inpl_adder=qrisp.gidney_adder)
G = [3, 2]

benchmark_results_ec_ctrl_add = {}

for anc_pair in anc_values:
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    anc[:] = anc_pair
    res_ec_ctrl_add = qECarithm.qrisp_ell_add_inpl(anc, G, p)
    bm = benchmark_circuit(res_ec_ctrl_add[0])
    benchmark_results_ec_ctrl_add[tuple(anc_pair)] = bm

In [9]:
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_ec_add, orient="index")
results_df.index.name = "Ancilla Values (anc)"
print(results_df)

{'x': 8050, 'cx': 123354, 'h': 34614, 't': 26115, 't_dg': 27996, 's': 9948, 'measure': 9948, 'c_if_cz': 9948, 'reset': 9948, 'p': 3240}
{'qubit_count': 72, 't_count': 54111, 'cnot_count': 128328.0, 't_depth': 12593}


In [ ]:
# # WARNING: different EC
# curve = clECarithm.EllCurve(a, b, p)

# x1 = qrisp.QuantumModulus(2**(p.bit_length())) 
# x2 = qrisp.QuantumModulus(2**(p.bit_length()))

# mod_p = qrisp.QuantumModulus(p)
# #store registers
# anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
# #initialize the store register to the point P_0 = [3,2]
# anc[:] = [3,2]

# #Generator G = [3,2]
# G = [3,2]

# #P = (3,3)
# P = [3,3]

# #Superposition
# qrisp.h(x1)
# qrisp.h(x2)

# #Multiplication step: anc = P0 + x1*G 
# anc = qECarithm.qrisp_ell_mult_add(G, anc, x1, curve)

# #Multiplication step: anc = anc - x2*P
# anc = qECarithm.qrisp_ell_mult_add(P, anc, x2, curve)

# #Quantum Fourier Transform on x1 and x2 registers
# qrisp.QFT(x1, inv=True)
# qrisp.QFT(x2, inv=True)

# Define the values of ancilla to test
anc_values = [
    (3, 2),
    (4, 2),
    (0, 5),
    (5, 0),
    (0, 2),
    (4, 5),
    (2, 1),
    (3, 5)
]

# Define the elliptic curve parameters and constants
curve = clECarithm.EllCurve(a, b, p)
x1 = qrisp.QuantumModulus(2**(p.bit_length()))
x2 = qrisp.QuantumModulus(2**(p.bit_length()))
mod_p = qrisp.QuantumModulus(p)
G = [3, 2]
P = [0, 2]

benchmark_results_mult_add = {}

for anc_pair in anc_values:
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    anc[:] = anc_pair

    qrisp.h(x1)
    qrisp.h(x2)
    anc = qECarithm.qrisp_ell_mult_add(G, anc, x1, curve)
    anc = qECarithm.qrisp_ell_mult_add(P, anc, x2, curve)
    qrisp.QFT(x1, inv=True)
    qrisp.QFT(x2, inv=True)

    bm = benchmark_circuit(anc[0])

    # Store results in the dictionary
    benchmark_results_mult_add[tuple(anc_pair)] = bm


In [ ]:
# Display results in a readable format
import pandas as pd
results_df = pd.DataFrame.from_dict(benchmark_results_mult_add, orient="index")
results_df.index.name = "Ancilla Values (anc)"
results_df.columns.name = "Benchmark Metrics"
print(results_df)

In [ ]:
# benchmark_results = {}

# benchmark_results["kaliski_quantum"] = benchmark_circuit(res_kaliski)
# benchmark_results["ec_addition"] = benchmark_circuit(res_ec_add[0])
# benchmark_results["controlled_addition"] = benchmark_circuit(res_ec_ctrl_add[0])
# benchmark_results["Shor's algorithm"] = benchmark_circuit(anc[0])

{'x': 593, 'cx': 6676, 'h': 2040, 't': 1666, 't_dg': 1840, 's': 548, 'measure': 548, 'c_if_cz': 548, 'reset': 548}
{'x': 2794, 'cx': 40842, 'h': 11697, 't': 9022, 't_dg': 9718, 's': 3371, 'measure': 3371, 'c_if_cz': 3371, 'reset': 3371}
{'x': 8050, 'cx': 123354, 'h': 34614, 't': 26115, 't_dg': 27996, 's': 9948, 'measure': 9948, 'c_if_cz': 9948, 'reset': 9948, 'p': 3240}


In [11]:
# import pandas as pd

# # Display results in a tabular format
# results_df = pd.DataFrame(benchmark_results).T
# print(results_df)

                     qubit_count  t_count  cnot_count  t_depth
kaliski_quantum             30.0   3506.0      6950.0    870.0
ec_addition                 70.0  18740.0     42527.5   4119.0
controlled_addition         72.0  54111.0    128328.0  12593.0
